In [52]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import re
from scipy import stats
import utils_multi as utils
import seaborn as sns
import pingouin as pg
import matplotlib.cm as cm

import statsmodels.stats.power as smp
from statsmodels.stats.anova import AnovaRM
from tqdm import tqdm


from natsort import index_natsorted

# plt.rcParams['font.family'] = 'Times New Roman'
# plt.rcParams['font.family'] = 'Calibri'

path_figs = "./Figs/"

fingers = ['1', '2', '3', '4', '5']

iti = 1000 # msecs for inter-trial interval
planTime = 2000 # msecs for precue time
feedbackTime = 2000 # msecs for feedback time


total_sub_num = 20
num_sessions = 1
num_blocks_per_session = 31
num_trials_per_block = 30

# sub_nums = [1,2]
sub_nums = [1]

utils.set_figure_style("1col")
sns.color_palette('colorblind')


[(0.00392156862745098, 0.45098039215686275, 0.6980392156862745),
 (0.8705882352941177, 0.5607843137254902, 0.0196078431372549),
 (0.00784313725490196, 0.6196078431372549, 0.45098039215686275),
 (0.8352941176470589, 0.3686274509803922, 0.0),
 (0.8, 0.47058823529411764, 0.7372549019607844),
 (0.792156862745098, 0.5686274509803921, 0.3803921568627451),
 (0.984313725490196, 0.6862745098039216, 0.8941176470588236),
 (0.5803921568627451, 0.5803921568627451, 0.5803921568627451),
 (0.9254901960784314, 0.8823529411764706, 0.2),
 (0.33725490196078434, 0.7058823529411765, 0.9137254901960784)]

In [53]:
# reload utils
import importlib
importlib.reload(utils)

<module 'utils_multi' from '/Users/amin/projects/LearningDynamics/ChordAdaptationDynamics/utils_multi.py'>

In [54]:
subjs_list = utils.read_dat_files_subjs_list(sub_nums)

subjs = pd.concat(subjs_list, ignore_index=True)

subjs['TotalTrialNum'] = (subjs['BN'] - 1) * num_trials_per_block + subjs['TN']
# subjs['day'] = subjs['BN'].apply(lambda x: (x - 1) // num_blocks_per_session + 1)
for i in range(1, 6):
    subjs[f'purturbation{i}'] = np.round(subjs[f'endForcePurturbed{i}'] - subjs[f'endForce{i}'], 2)

subjs['num_targets'] = subjs[[f'targetForce{i}' for i in range(1, 6)]].ne(0).sum(axis=1)
subjs.reset_index(drop=True, inplace=True)



In [46]:
subjs.columns

Index(['BN', 'TN', 'subNum', 'targetForce1', 'targetForce2', 'targetForce3',
       'targetForce4', 'targetForce5', 'endForce1', 'endForce2', 'endForce3',
       'endForce4', 'endForce5', 'endForcePurturbed1', 'endForcePurturbed2',
       'endForcePurturbed3', 'endForcePurturbed4', 'endForcePurturbed5',
       'planTime', 'feedbackTime', 'iti', 'forceGain', 'trialCorr',
       'trialErrorType', 'TotalTrialNum', 'purturbation1', 'purturbation2',
       'purturbation3', 'purturbation4', 'purturbation5', 'num_targets'],
      dtype='str')

In [47]:
subjs = subjs[['subNum', 'BN', 'TN', 'TotalTrialNum', 'targetForce1', 'targetForce2', 'targetForce3', 'targetForce4', 'targetForce5',
            'endForce1', 'endForce2', 'endForce3', 'endForce4', 'endForce5', 
            'endForcePurturbed1', 'endForcePurturbed2', 'endForcePurturbed3', 'endForcePurturbed4', 'endForcePurturbed5',
            'purturbation1', 'purturbation2', 'purturbation3', 'purturbation4', 'purturbation5',
            'forceGain','trialCorr', 'trialErrorType', 'num_targets']]

In [48]:
# subjs_force = pd.read_csv(utils.path_misc+'subjs_force_train_exp1.csv', sep = '\t')


In [49]:
# # for missing values of endForce, extract from subjs_force
# for i in range(1,6):
#     subjs[f'endForce{i}'] = subjs.apply(lambda row: subjs_force.loc[(subjs_force['subNum'] == row['subNum']) & 
#                                                                     (subjs_force['BN'] == row['BN']) & 
#                                                                     (subjs_force['TN'] == row['TN']), f'endForce{i}'].values[0] 
#                                         if pd.isna(row[f'endForce{i}']) else row[f'endForce{i}'], axis=1)

In [51]:
subjs

,subNum,BN,TN,TotalTrialNum,targetForce1,targetForce2,targetForce3,targetForce4,targetForce5,endForce1,...,endForcePurturbed5,purturbation1,purturbation2,purturbation3,purturbation4,purturbation5,forceGain,trialCorr,trialErrorType,num_targets
0,1,1,1,1,-2.0,-2.0,0.0,-2.0,-2.0,-6.277439e+66,...,-6.277439e+66,0.00,0.00,0.00,0.00,0.00,1.0,0,2,4
1,1,1,2,2,-2.0,-2.0,0.0,-2.0,-2.0,-4.056000e+00,...,-2.408000e+00,-0.94,0.79,0.32,0.24,-0.38,1.0,1,0,4
2,1,1,3,3,-2.0,-2.0,0.0,-2.0,-2.0,-1.075000e+00,...,-3.870000e-01,-0.32,0.53,0.36,0.24,0.34,1.0,1,0,4
3,1,1,4,4,-2.0,-2.0,0.0,-2.0,-2.0,-6.277439e+66,...,-6.277439e+66,0.00,0.00,0.00,0.00,0.00,1.0,0,2,4
4,1,1,5,5,-2.0,-2.0,0.0,-2.0,-2.0,-1.654000e+00,...,-1.138000e+00,0.38,-0.50,0.38,-0.66,0.21,1.0,1,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
445,1,15,26,446,0.0,-2.0,0.0,0.0,-2.0,-1.790000e-01,...,-1.834000e+00,-0.13,0.41,0.72,0.27,0.34,1.0,1,0,2
446,1,15,27,447,0.0,-2.0,0.0,0.0,-2.0,-2.060000e-01,...,-1.737000e+00,-0.16,-0.32,0.44,-0.40,-0.09,1.0,1,0,2
447,1,15,28,448,0.0,-2.0,0.0,0.0,-2.0,-1.900000e-01,...,-1.442000e+00,-0.68,-0.07,0.67,-0.26,0.03,1.0,1,0,2
448,1,15,29,449,0.0,-2.0,0.0,0.0,-2.0,-2.430000e-01,...,-1.471000e+00,-0.44,-0.01,0.56,-0.88,-0.06,1.0,1,0,2


In [44]:
# aligned_cut_force.to_csv(utils.path_misc+'aligned_cut_force.csv', index = False, sep = '\t')

subjs.to_csv(utils.path_misc+'subjs.csv', index = False, sep = '\t')